In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

BRONZE = "olist_project.bronze"
SILVER = "olist_project.silver"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER} COMMENT 'Cleansed and typed layer'")

DataFrame[]

In [0]:
orders_df = spark.table(f"{BRONZE}.orders")
silver_orders = orders_df \
   .withColumn("order_purchase_timestamp",    F.to_timestamp("order_purchase_timestamp")) \
   .withColumn("order_approved_at",           F.to_timestamp("order_approved_at")) \
   .withColumn("order_delivered_carrier_date",F.to_timestamp("order_delivered_carrier_date")) \
   .withColumn("order_delivered_customer_date",F.to_timestamp("order_delivered_customer_date")) \
   .withColumn("order_estimated_delivery_date",F.to_timestamp("order_estimated_delivery_date")) \
   .withColumn("order_status", F.trim(F.lower("order_status"))) \
   .dropDuplicates(["order_id"]) \
   .filter(F.col("order_id").isNotNull())
silver_orders.write \
   .format("delta") \
   .mode("overwrite") \
   .option("overwriteSchema", "true") \
   .saveAsTable(f"{SILVER}.orders")
print(f"silver.orders — {silver_orders.count()} rows")

silver.orders — 99441 rows


In [0]:
customers_df = spark.table(f"{BRONZE}.customers")
silver_customers = customers_df \
   .withColumnRenamed("customer_zip_code_prefix", "zip_code") \
   .withColumn("customer_city",  F.initcap(F.trim("customer_city"))) \
   .withColumn("customer_state", F.upper(F.trim("customer_state"))) \
   .dropDuplicates(["customer_id"]) \
   .filter(F.col("customer_id").isNotNull())
silver_customers.write \
   .format("delta") \
   .mode("overwrite") \
   .option("overwriteSchema", "true") \
   .saveAsTable(f"{SILVER}.customers")
print(f"silver.customers — {silver_customers.count()} rows")

silver.customers — 99441 rows


In [0]:
items_df = spark.table(f"{BRONZE}.order_items")
silver_items = items_df \
   .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date")) \
   .withColumn("price",         F.col("price").cast(DecimalType(10, 2))) \
   .withColumn("freight_value", F.col("freight_value").cast(DecimalType(10, 2))) \
   .filter(F.col("order_id").isNotNull()) \
   .filter(F.col("price") > 0)
silver_items.write \
   .format("delta") \
   .mode("overwrite") \
   .option("overwriteSchema", "true") \
   .saveAsTable(f"{SILVER}.order_items")
print(f"silver.order_items — {silver_items.count()} rows")

silver.order_items — 112650 rows


In [0]:
payments_df = spark.table(f"{BRONZE}.order_payments")
silver_payments = payments_df \
   .withColumn("payment_value",        F.col("payment_value").cast(DecimalType(10, 2))) \
   .withColumn("payment_installments", F.col("payment_installments").cast(IntegerType())) \
   .withColumn("payment_type",         F.trim(F.lower("payment_type"))) \
   .filter(F.col("order_id").isNotNull()) \
   .filter(F.col("payment_value") >= 0)
silver_payments.write \
   .format("delta") \
   .mode("overwrite") \
   .option("overwriteSchema", "true") \
   .saveAsTable(f"{SILVER}.order_payments")
print(f"silver.order_payments — {silver_payments.count()} rows")

silver.order_payments — 103886 rows


In [0]:
reviews_df = spark.table(f"{BRONZE}.order_reviews")
silver_reviews = reviews_df \
   .withColumn("review_creation_date",  F.to_timestamp("review_creation_date")) \
   .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp")) \
   .withColumn("review_score", F.col("review_score").cast(IntegerType())) \
   .drop("review_comment_title", "review_comment_message") \
   .dropDuplicates(["review_id"]) \
   .filter(F.col("order_id").isNotNull()) \
   .filter(F.col("review_score").between(1, 5))
silver_reviews.write \
   .format("delta") \
   .mode("overwrite") \
   .option("overwriteSchema", "true") \
   .saveAsTable(f"{SILVER}.order_reviews")
print(f"silver.order_reviews — {silver_reviews.count()} rows")

silver.order_reviews — 98410 rows


In [0]:
products_df  = spark.table(f"{BRONZE}.products")
category_df  = spark.table(f"{BRONZE}.category_translation")
silver_products = products_df \
   .join(category_df, on="product_category_name", how="left") \
   .withColumnRenamed("product_category_name_english", "category_name_english") \
   .withColumn("product_weight_g",  F.col("product_weight_g").cast(IntegerType())) \
   .withColumn("product_length_cm", F.col("product_length_cm").cast(IntegerType())) \
   .withColumn("product_height_cm", F.col("product_height_cm").cast(IntegerType())) \
   .withColumn("product_width_cm",  F.col("product_width_cm").cast(IntegerType())) \
   .drop("product_category_name") \
   .dropDuplicates(["product_id"]) \
   .filter(F.col("product_id").isNotNull())
silver_products.write \
   .format("delta") \
   .mode("overwrite") \
   .option("overwriteSchema", "true") \
   .saveAsTable(f"{SILVER}.products")
print(f"silver.products — {silver_products.count()} rows")

silver.products — 32951 rows


In [0]:
sellers_df = spark.table(f"{BRONZE}.sellers")
silver_sellers = sellers_df \
   .withColumnRenamed("seller_zip_code_prefix", "zip_code") \
   .withColumn("seller_city",  F.initcap(F.trim("seller_city"))) \
   .withColumn("seller_state", F.upper(F.trim("seller_state"))) \
   .dropDuplicates(["seller_id"]) \
   .filter(F.col("seller_id").isNotNull())
silver_sellers.write \
   .format("delta") \
   .mode("overwrite") \
   .option("overwriteSchema", "true") \
   .saveAsTable(f"{SILVER}.sellers")
print(f"silver.sellers — {silver_sellers.count()} rows")

silver.sellers — 3095 rows


In [0]:
silver_tables = ["orders", "customers", "order_items", "order_payments",
                "order_reviews", "products", "sellers"]
print(f"{'Table':<20} {'Row Count':>10}")
print("-" * 32)
for t in silver_tables:
   count = spark.table(f"{SILVER}.{t}").count()
   print(f"{t:<20} {count:>10,}")

Table                 Row Count
--------------------------------
orders                   99,441
customers                99,441
order_items             112,650
order_payments          103,886
order_reviews            98,410
products                 32,951
sellers                   3,095
